# 01 · Raw data profiling (HI-Small)

**Goal:** collect the facts needed to define the ingestion schema and the validation rules (phase 1).
Reads the raw CSV read-only via DuckDB. The EDA proper (phase 2) runs on the processed Parquet in `02_eda.ipynb`.

Every section ends with a **Finding** and the **Decision** that follows from it.

## 0 · Setup

In [47]:
from pathlib import Path

import duckdb
import pandas as pd

# Locate the repo root so the notebook works regardless of the working directory
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRANS_CSV = RAW_DIR / "HI-Small_Trans.csv"
PATTERNS_TXT = RAW_DIR / "HI-Small_Patterns.txt"
ACCOUNTS_CSV = RAW_DIR / "HI-Small_accounts.csv"

con = duckdb.connect()  # in-memory, nothing is persisted
TRANS_CSV.exists(), PATTERNS_TXT.exists(), ACCOUNTS_CSV.exists()

(True, True, True)

In [48]:
# Raw-text view: every column as VARCHAR, so we profile the file, not a parser's interpretation
con.read_csv(str(TRANS_CSV), all_varchar=True).create_view("raw_text", replace=True)

TimestampVARCHAR,From BankVARCHAR,AccountVARCHAR,To BankVARCHAR,Account_1VARCHAR,Amount ReceivedVARCHAR,Receiving CurrencyVARCHAR,Amount PaidVARCHAR,Payment CurrencyVARCHAR,Payment FormatVARCHAR,Is LaunderingVARCHAR
2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
2022/09/01 00:02,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
2022/09/01 00:06,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
2022/09/01 00:03,001,8000F5AD0,001,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
2022/09/01 00:08,001,8000EBAC0,001,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
2022/09/01 00:16,001,8000EC1E0,001,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
2022/09/01 00:26,012,8000EC280,002439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
2022/09/01 00:21,001,8000EDEC0,0211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0


## 1 · File structure

- How many columns, and what does the header look like (duplicate `Account`)?
- How many data rows?
- What does DuckDB's sniffer guess per column? (Only for exploring, never for ingestion.)

In [49]:
# IDs and categories as string: integer parsing would drop leading zeros ("010" -> 10).
# Without explicit names, pandas renames the duplicate "Account" header to "Account.1".
transaction = pd.read_csv(
    TRANS_CSV,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string",
        "Receiving Currency": "string",
        "Payment Currency": "string",
        "Payment Format": "string",
    },
)

print(f"rows: {len(transaction):,} | columns: {transaction.shape[1]}")
transaction.head()

rows: 5,078,345 | columns: 11


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [50]:
# What DuckDB's sniffer would guess (exploration only, never used for ingestion).
# fetchall() closes the result; fetchone() would leave it open and block the connection.
sniff_result = con.execute("SELECT Columns FROM sniff_csv(?)", [str(TRANS_CSV)]).fetchall()
sniffed_columns = sniff_result[0][0]

for column in sniffed_columns:
    print(f"{column['name']}: {column['type']}")

Timestamp: TIMESTAMP
From Bank: VARCHAR
Account: VARCHAR
To Bank: VARCHAR
Account_1: VARCHAR
Amount Received: DOUBLE
Receiving Currency: VARCHAR
Amount Paid: DOUBLE
Payment Currency: VARCHAR
Payment Format: VARCHAR
Is Laundering: BIGINT


**Finding:** 5,078,345 data rows, 11 columns. The header contains `Account` twice (sender and receiver account): pandas renames the second one to `Account.1`, DuckDB to `Account_1`. The sniffer guesses amounts as `DOUBLE` and the label as `BIGINT`.

**Decision:** Ingestion uses its own snake_case column names (`from_account`, `to_account`, …) and an explicit schema with `auto_detect=false`. Sniffer output is never used for ingestion.

## 2 · Amounts

- Min and max of `Amount Received` and `Amount Paid`. Does `DECIMAL(18,2)` fit?
- Are there ever more than 2 decimal places?
- Zero or negative amounts?

In [51]:
# Min/max as parsed floats (no rounding: round(..., 2) showed 0.0 for the 0.000001 minimum)
transaction[["Amount Received", "Amount Paid"]].agg(["min", "max"])

,Amount Received,Amount Paid
min,1.000000e-06,1.000000e-06
max,1.046302e+12,1.046302e+12


In [52]:
# Digits before/after the decimal point, measured on the raw text of both amount columns
amounts_raw = pd.read_csv(TRANS_CSV, usecols=["Amount Received", "Amount Paid"], dtype=str)

digits = pd.DataFrame(
    {
        column: {
            "max_integer_digits": parts[0].str.len().max(),
            "max_decimal_places": parts[1].str.len().max(),
        }
        for column in amounts_raw.columns
        for parts in [amounts_raw[column].str.split(".", expand=True)]
    }
)
digits

,Amount Received,Amount Paid
max_integer_digits,13,13
max_decimal_places,6,6


In [53]:
# Amounts measured on raw text (float -> str would turn small values into "1e-06")
con.sql("""
    SELECT
        min("Amount Paid"::DOUBLE)                         AS min_paid,
        max("Amount Paid"::DOUBLE)                         AS max_paid,
        count(*) FILTER ("Amount Paid"::DOUBLE <= 0)       AS paid_le_zero,
        count(*) FILTER ("Amount Received"::DOUBLE <= 0)   AS received_le_zero,
        max(length(split_part("Amount Paid", '.', 1)))     AS max_int_digits,
        max(length(split_part("Amount Paid", '.', 2)))     AS max_decimals
    FROM raw_text
""").df()

,min_paid,max_paid,paid_le_zero,received_le_zero,max_int_digits,max_decimals
0,0.000001,1.046302e+12,0,0,13,6


**Finding:** Both amount columns range from 0.000001 to 1,046,302,363,293.48: at most 13 integer digits and 6 decimal places (only Bitcoin uses more than 2). There are no zero or negative amounts. The `0.0` minimum above is an artefact of `round(..., 2)`. Measured on raw text, because float → str turns small values into `1e-06`.

**Decision:** `DECIMAL(19,6)` for both amount columns. Validation fails on amounts ≤ 0.

## 3 · Categorical values

- Distinct currencies (receiving vs. payment). Do the two sets differ?
- Distinct payment formats.
- Bank IDs: lengths, leading zeros, distinct count.

In [54]:
# Currency distribution (payment side)
transaction["Payment Currency"].value_counts()

Payment Currency
US Dollar            1895172
Euro                 1168297
Swiss Franc           234860
Yuan                  213752
Shekel                192184
Rupee                 190202
UK Pound              180738
Yen                   155209
Ruble                 155178
Bitcoin               146066
Canadian Dollar       140042
Australian Dollar     136769
Mexican Peso          110159
Saudi Riyal            89014
Brazil Real            70703
Name: count, dtype: Int64

In [55]:
# Do payment and receiving currencies use the same set of values?
payment_currencies = set(transaction["Payment Currency"])
receiving_currencies = set(transaction["Receiving Currency"])

print("Only in Payment Currency:", payment_currencies - receiving_currencies)
print("Only in Receiving Currency:", receiving_currencies - payment_currencies)

Only in Payment Currency: set()
Only in Receiving Currency: set()


In [56]:
# Payment formats
transaction["Payment Format"].unique()

<StringArray>
['Reinvestment', 'Cheque', 'Credit Card', 'ACH', 'Cash', 'Wire', 'Bitcoin']
Length: 7, dtype: string

In [57]:
# Bank IDs as text: cardinality and most frequent value
transaction["From Bank"].describe()

count     5078345
unique      30528
top           070
freq       449859
Name: From Bank, dtype: object

In [58]:
transaction["To Bank"].describe()

count     5078345
unique      15850
top           012
freq        34732
Name: To Bank, dtype: object

In [59]:
# Share of receiving bank IDs with a leading zero
has_leading_zero = transaction["To Bank"].str.startswith("0")
has_leading_zero.mean()

np.float64(1.0)

In [60]:
transaction["To Bank"].nunique()

15850

In [61]:
# Bank IDs across both columns: lengths, leading zeros, collisions after stripping zeros
con.sql("""
    WITH banks AS (
        SELECT "From Bank" AS bank FROM raw_text
        UNION ALL
        SELECT "To Bank" FROM raw_text
    )
    SELECT
        min(length(bank))                    AS min_len,
        max(length(bank))                    AS max_len,
        avg(starts_with(bank, '0')::INT)     AS share_leading_zero,
        count(DISTINCT bank)                 AS distinct_as_text,
        count(DISTINCT ltrim(bank, '0'))     AS distinct_as_integer
    FROM banks
""").df()

,min_len,max_len,share_leading_zero,distinct_as_text,distinct_as_integer
0,3,7,1.0,30528,30470


**Finding:** 15 currencies, identical in both currency columns. 72,170 rows have different payment and receiving currencies (FX transfers). 7 payment formats. Bank IDs have 3–7 characters and all start with `0`. Across both columns there are 30,528 distinct IDs as text but only 30,470 after stripping leading zeros (e.g. `0015` vs. `015`), so integer parsing would merge different banks. Payment format `Bitcoin` does not strictly imply currency `Bitcoin` (78 rows with format `Bitcoin` and non-Bitcoin currencies).

**Decision:** Bank and account IDs as `VARCHAR`. Validation checks currencies and payment formats against fixed allowlists. No hard rule links payment format and currency.

## 4 · Time coverage

- First and last timestamp. Does every value parse with `%Y/%m/%d %H:%M`?
- Rows per day: gaps, or an incomplete first or last day?

In [62]:
# Parse with the explicit format: any value that does not match raises instead of being guessed
timestamps = pd.to_datetime(transaction["Timestamp"], format="%Y/%m/%d %H:%M")
print("first:", timestamps.min(), "| last:", timestamps.max())

# Rows per calendar day, in time order
timestamps.dt.floor("D").value_counts().sort_index()

first: 2022-09-01 00:00:00 | last: 2022-09-18 16:18:00


Timestamp
2022-09-01    1114921
2022-09-02     754449
2022-09-03     207382
2022-09-04     207430
2022-09-05     482650
2022-09-06     482089
2022-09-07     482751
2022-09-08     482773
2022-09-09     654467
2022-09-10     208325
2022-09-11        396
2022-09-12        281
2022-09-13        184
2022-09-14        121
2022-09-15         46
2022-09-16         46
2022-09-17         23
2022-09-18         11
Name: count, dtype: int64

In [63]:
# Label rate before vs. after 2022-09-11 (low-volume tail)
con.sql("""
    SELECT
        strptime("Timestamp", '%Y/%m/%d %H:%M') >= TIMESTAMP '2022-09-11' AS is_tail,
        count(*)                                                         AS n_rows,
        avg("Is Laundering"::INT)                                        AS label_rate
    FROM raw_text
    GROUP BY ALL
    ORDER BY is_tail
""").df()

,is_tail,n_rows,label_rate
0,False,5077237,0.000891
1,True,1108,0.591155


**Finding:** All timestamps parse with `%Y/%m/%d %H:%M` (minute resolution). The file is not sorted by time. 2022-09-01 to 2022-09-10 is dense (≈ 200k–1.1M rows per day, Saturdays and Sundays ≈ 207k). 2022-09-11 to 2022-09-18 contains only 1,108 rows, with a label rate of ≈ 59 % (vs. ≈ 0.09 % before). These are most likely laundering patterns that run past the end of the simulation.

**Decision:** Ingestion keeps all rows; raw data is not filtered. How to treat the tail (cut it off or use it as part of the test period) is decided in phase 2 together with the split date.

## 5 · Label

- Values of `Is Laundering` (only 0 and 1?).
- Overall label rate.

In [64]:
# Label values and rate
transaction["Is Laundering"].value_counts().to_frame("count").assign(
    share=transaction["Is Laundering"].value_counts(normalize=True)
)

,count,share
Is Laundering,,
0,5073168,0.998981
1,5177,0.001019


**Finding:** The label only takes the values 0 and 1. 5,177 positives out of 5,078,345 rows, so the label rate is ≈ 0.102 % (≈ 1 in 980).

**Decision:** Store the label as `BOOLEAN`. Accuracy and ROC-AUC are unsuitable as main metrics; we use PR-AUC and Precision@k.

## 6 · Nulls and duplicates

- Null or empty values per column.
- Fully identical rows. Are they real repeats or errors? What would be the key of a transaction?

In [65]:
# Nulls per column
transaction.isna().sum().sort_values(ascending=False)

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

In [66]:
# Nulls plus empty or whitespace-only strings per column
text_cols = transaction.select_dtypes(include=["object", "string"]).columns

empty = transaction[text_cols].apply(lambda col: col.str.fullmatch(r"\s*", na=False))

overview = (
    pd.DataFrame({"missing": transaction.isna().sum(), "empty_string": empty.sum()})
    .fillna(0)
    .astype(int)
)

overview["total_missing_or_empty"] = overview["missing"] + overview["empty_string"]
overview["percent"] = (overview["total_missing_or_empty"] / len(transaction) * 100).round(2)

overview.sort_values("total_missing_or_empty", ascending=False)

,missing,empty_string,total_missing_or_empty,percent
Account,0,0,0,0.0
Account.1,0,0,0,0.0
Amount Paid,0,0,0,0.0
Amount Received,0,0,0,0.0
From Bank,0,0,0,0.0
Is Laundering,0,0,0,0.0
Payment Currency,0,0,0,0.0
Payment Format,0,0,0,0.0
Receiving Currency,0,0,0,0.0
Timestamp,0,0,0,0.0


In [67]:
# Fully identical rows (all 11 columns equal)
transaction.duplicated().sum()

np.int64(9)

**Finding:** No nulls and no empty strings in any column. 9 rows are exact copies of another row (e.g. two Bitcoin transfers of 0.000008 in the same minute between the same accounts). With minute resolution and no transaction ID in the source, these are plausibly genuine repeated transfers rather than errors. There is no natural key.

**Decision:** Keep the duplicates. Ingestion adds a surrogate `transaction_id` (row number in the raw file) as the key. Validation fails on nulls and logs the duplicate count without failing.